In [ ]:
import pandas as pd
import numpy as np

In [ ]:
N = 5000
SEED = 42

rng = np.random.default_rng(SEED)

In [ ]:
df = pd.DataFrame()

df["request_id"] = np.arange(10001, 10001 + N)

Тип закупки

In [ ]:
df["purchase_type"] = rng.choice(
    ["Простая", "IT", "Юридическая"],
    size=N,
    p=[0.50, 0.35, 0.15]
)

In [ ]:
df["department"] = rng.choice(
    [
        "IT",
        "Marketing",
        "Sales",
        "HR",
        "Finance",
        "Operations",
        "Administration"
    ],
    size=N,
    p=[0.20, 0.15, 0.20, 0.10, 0.10, 0.15, 0.10]
)

Генерируем даты

In [ ]:
start_date = pd.Timestamp("2026-01-01")
end_date = pd.Timestamp("2026-12-31")

days = (end_date - start_date).days

random_days = rng.integers(
    0,
    days + 1,
    size=N
)

df["request_date"] = (
    start_date +
    pd.to_timedelta(random_days, unit="D")
)


Генерируем стоимости

In [ ]:
amounts = []
for purchase_type in df["purchase_type"]:
    if purchase_type == "Простая":
        amount = rng.lognormal(
            mean=np.log(15000),
            sigma=0.65
        )

        amount = np.clip(
            amount,
            2000,
            50000
        )

    elif purchase_type == "IT":

        amount = rng.lognormal(
            mean=np.log(100000),
            sigma=0.55
        )

        amount = np.clip(
            amount,
            30000,
            300000
        )

    else:

        amount = rng.lognormal(
            mean=np.log(400000),
            sigma=0.75
        )

        amount = np.clip(
            amount,
            100000,
            2000000
        )

    amounts.append(round(amount, 2))


df["amount"] = amounts

Строим маршруты

In [ ]:
routes = {

    "Простая":
        "Manager → Procurement",

    "IT":
        "Manager → IT → Finance → Procurement",

    "Юридическая":
        "Manager → Finance → Legal → Procurement"
}


df["approval_route"] = df["purchase_type"].map(routes)

Возвраты

In [ ]:
return_probabilities = {

    "Простая": 0.05,

    "IT": 0.25,

    "Юридическая": 0.20
}


return_count = []

for purchase_type in df["purchase_type"]:
    probability = return_probabilities[purchase_type]
    if rng.random() < probability:

        count = rng.choice(
            [1, 2, 3],
            p=[0.70, 0.23, 0.07]
        )

    else:

        count = 0
    return_count.append(count)


df["return_count"] = return_count

Добавляем уточнения

In [ ]:
clarification_count = []
for returns in df["return_count"]:

    if returns == 0:

        count = rng.choice(
            [0, 1],
            p=[0.70, 0.30]
        )

    elif returns == 1:

        count = rng.integers(1, 4)

    else:

        count = rng.integers(3, 7)

    clarification_count.append(count)


df["clarification_count"] = clarification_count

# Также добавим отклонения от маршрута: 5% заявок идут не по стандартному маршруту

for i in range(N):
    if rng.random() < 0.05:

        purchase_type = df.loc[i, "purchase_type"]

        if purchase_type == "IT":

            df.loc[i, "approval_route"] = rng.choice([
                "Manager → Finance → IT → Procurement",
                "Manager → IT → Procurement → Finance"
            ])

        elif purchase_type == "Юридическая":

            df.loc[i, "approval_route"] = rng.choice([
                "Manager → Legal → Finance → Procurement",
                "Manager → Finance → Procurement → Legal"
            ])

        else:

            df.loc[i, "approval_route"] = (
                "Manager → Finance → Procurement"
            )


Время обработки заявки

In [ ]:
mean_time = {

    "Простая": 8,

    "IT": 30,

    "Юридическая": 55
}

std_time = {

    "Простая": 5,

    "IT": 14,

    "Юридическая": 25
}

total_time = []
for i in range(N):
    purchase_type = df.loc[i, "purchase_type"]

    # Базовое время
    time = rng.normal(
        mean_time[purchase_type],
        std_time[purchase_type]
    )

    # Время не может быть меньше 1 часа
    time = max(time, 1)


    # Возвраты увеличивают время
    time += (
        df.loc[i, "return_count"]
        * rng.uniform(5, 15)
    )


    # Уточнения увеличивают время
    time += (
        df.loc[i, "clarification_count"]
        * rng.uniform(0.5, 2)
    )


    # Нестандартный маршрут увеличивает время
    if df.loc[i, "approval_route"] != routes[purchase_type]:

        time += rng.uniform(5, 20)

    total_time.append(round(time, 2))

df["total_time_hours"] = total_time


Статусы: в процессе, одобрено, отклонено

In [ ]:
statuses = []
rejection_reasons = []

for i in range(N):

    purchase_type = df.loc[i, "purchase_type"]

    random_value = rng.random()

    if random_value < 0.08:
        status = "В процессе"
        reason = None
    elif random_value < 0.16:
        status = "Отклонена"

        if purchase_type == "IT":
            reason = rng.choice([
                "Технические требования не соблюдены",
                "Недостаточно бюджета",
                "Некорректные данные"
            ])

        elif purchase_type == "Юридическая":

            reason = rng.choice([
                "Юридические риски",
                "Недостаточно бюджета",
                "Некорректные данные"
            ])
        else:

            reason = rng.choice([
                "Недостаточно бюджета",
                "Некорректные данные",
                "Не согласовано руководителем"
            ])

    else:

        status = "Одобрена"
        reason = None

    statuses.append(status)
    rejection_reasons.append(reason)


df["final_status"] = statuses
df["rejection_reason"] = rejection_reasons

In [ ]:
df = df[
    [
        "request_id",
        "purchase_type",
        "amount",
        "department",
        "request_date",
        "approval_route",
        "total_time_hours",
        "return_count",
        "rejection_reason",
        "clarification_count",
        "final_status"
    ]
]


In [ ]:
df = df.sort_values(
    "request_date"
).reset_index(drop=True)

In [ ]:
from google.colab import files
df.to_csv("procurement_requests_5000.csv", index=False, encoding="utf-8-sig")
files.download("procurement_requests_5000.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>